# Resume Llama 3 LoRA — Untouched Holdout Evaluation

This notebook evaluates the trained PEFT adapter before merge or GGUF conversion. It uses only the first two holdout turns as model input; the assistant turn is retained solely as an evaluation label. Progress is saved to Google Drive after every record.

Manual inputs: select a **T4 GPU**, then upload `resume-llama3-lora-adapter.zip` and `holdout_eval.jsonl` when prompted. Do not upload `training_data.jsonl`.

In [ ]:
%pip install -q --upgrade "transformers>=5.17,<6" "peft>=0.21,<1" "accelerate>=1.15,<2" "bitsandbytes>=0.50,<1" "huggingface_hub>=1,<2"

In [ ]:
from pathlib import Path
from google.colab import drive, files
from zipfile import ZipFile
from datetime import datetime, timezone
from importlib.metadata import version
import hashlib
import json
import shutil
import struct
import time
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select Runtime > Change runtime type > T4 GPU.")
gpu = torch.cuda.get_device_properties(0)
if gpu.total_memory / 1024**3 < 14:
    raise RuntimeError(f"At least 14 GiB VRAM is required; detected {gpu.total_memory / 1024**3:.1f} GiB.")
print(f"GPU: {gpu.name}; VRAM: {gpu.total_memory / 1024**3:.1f} GiB")
drive.mount('/content/drive')
WORK_DIR = Path('/content/drive/MyDrive/resume-builder-adapter-eval')
WORK_DIR.mkdir(parents=True, exist_ok=True)
PROGRESS_PATH = WORK_DIR / 'adapter_holdout_progress.json'

## Upload and validate the adapter and untouched holdout

Both files are pinned by SHA-256. The adapter ZIP is checked for corruption and unsafe paths before extraction.

In [ ]:
EXPECTED_ADAPTER_SHA256 = 'eaf9efd5c0f29421d17e9a4a87a546f671f4913b2551addeb1e15c18790982c1'
EXPECTED_HOLDOUT_SHA256 = '8d25b49f6d706254b61f9361e56592185b674a79af504175bcf65888484e1d72'
EXPECTED_TRAINING_SHA256 = '72baab567553c43ec815db089905d23f108978a50039cffeb18e4bfc0d64d70d'
BASE_MODEL = 'meta-llama/Meta-Llama-3-8B-Instruct'
BASE_REVISION = '8afb486c1db24fe5011ec46dfbe5b5dccdb575c2'
ADAPTER_ZIP = Path('/content/resume-llama3-lora-adapter.zip')
HOLDOUT_PATH = Path('/content/holdout_eval.jsonl')
ADAPTER_DIR = Path('/content/resume-llama3-lora-adapter')

uploaded = files.upload()
for filename, content in uploaded.items():
    Path('/content', filename).write_bytes(content)
if not ADAPTER_ZIP.exists() or not HOLDOUT_PATH.exists():
    raise FileNotFoundError('Upload resume-llama3-lora-adapter.zip and holdout_eval.jsonl.')

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

adapter_sha256 = sha256(ADAPTER_ZIP)
holdout_sha256 = sha256(HOLDOUT_PATH)
if adapter_sha256 != EXPECTED_ADAPTER_SHA256:
    raise ValueError(f'Adapter hash mismatch: {adapter_sha256}')
if holdout_sha256 != EXPECTED_HOLDOUT_SHA256:
    raise ValueError(f'Holdout hash mismatch: {holdout_sha256}')

required_files = {'adapter_model.safetensors', 'adapter_config.json', 'training_manifest.json', 'tokenizer.json', 'tokenizer_config.json'}
with ZipFile(ADAPTER_ZIP) as archive:
    names = set(archive.namelist())
    if archive.testzip() is not None:
        raise ValueError('Adapter ZIP failed its CRC check.')
    if required_files - names:
        raise ValueError(f'Adapter ZIP is missing: {sorted(required_files - names)}')
    if any(Path(name).is_absolute() or '..' in Path(name).parts for name in names):
        raise ValueError('Adapter ZIP contains an unsafe path.')
    manifest = json.loads(archive.read('training_manifest.json'))
    if manifest.get('base_model') != BASE_MODEL or manifest.get('base_revision') != BASE_REVISION:
        raise ValueError('Adapter base model or revision does not match the approved checkpoint.')
    if manifest.get('dataset_sha256') != EXPECTED_TRAINING_SHA256 or manifest.get('untouched_holdout_records_used') != 0:
        raise ValueError('Adapter manifest failed the training-data isolation check.')
    if ADAPTER_DIR.exists():
        shutil.rmtree(ADAPTER_DIR)
    ADAPTER_DIR.mkdir()
    archive.extractall(ADAPTER_DIR)

records = [json.loads(line) for line in HOLDOUT_PATH.read_text(encoding='utf-8-sig').splitlines() if line.strip()]
if len(records) != 30:
    raise ValueError(f'Expected exactly 30 holdout records; found {len(records)}.')
for index, record in enumerate(records):
    roles = tuple(message.get('role') for message in record.get('messages', []))
    if roles != ('system', 'user', 'assistant'):
        raise ValueError(f'Holdout record {index} has invalid roles: {roles}')
print('Artifact and untouched holdout validation: PASS')
print(f'Adapter SHA-256: {adapter_sha256}')
print(f'Holdout SHA-256: {holdout_sha256}')

## Authenticate and load the exact base revision plus adapter

In [ ]:
from huggingface_hub import hf_hub_download, notebook_login
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

notebook_login()
hf_hub_download(repo_id=BASE_MODEL, revision=BASE_REVISION, filename='config.json')
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, revision=BASE_REVISION, quantization_config=quantization, device_map={'': 0}, dtype=compute_dtype,
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
model.config.use_cache = True
print(f'Loaded adapter on exact base revision; CUDA allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB')

## Run resumable inference over all 30 records

Expected assistant responses are never included in the prompt. A progress file is atomically updated in Drive after each record. Re-run this cell after a disconnect to continue.

In [ ]:
def record_id(record):
    raw = json.dumps(record['messages'][:2], sort_keys=True, ensure_ascii=False).encode('utf-8')
    return hashlib.sha256(raw).hexdigest()[:16]

def extract_json(text):
    cleaned = text.strip()
    if cleaned.startswith('```'):
        cleaned = cleaned.split('\n', 1)[-1]
        if cleaned.endswith('```'):
            cleaned = cleaned[:-3].strip()
    try:
        value = json.loads(cleaned)
        return value if isinstance(value, dict) else None
    except json.JSONDecodeError:
        start, end = cleaned.find('{'), cleaned.rfind('}')
        if start < 0 or end <= start:
            return None
        try:
            value = json.loads(cleaned[start:end + 1])
            return value if isinstance(value, dict) else None
        except json.JSONDecodeError:
            return None

metadata = {
    'adapter_sha256': adapter_sha256,
    'holdout_sha256': holdout_sha256,
    'base_model': BASE_MODEL,
    'base_revision': BASE_REVISION,
    'generation_options': {'temperature': 0.15, 'top_p': 0.9, 'seed': 42, 'max_new_tokens': 1800},
}
progress = dict(metadata, results=[])
if PROGRESS_PATH.exists():
    existing = json.loads(PROGRESS_PATH.read_text(encoding='utf-8'))
    if any(existing.get(key) != value for key, value in metadata.items()):
        raise ValueError('Saved progress belongs to a different artifact, holdout, base, or generation configuration.')
    progress = existing
results_by_id = {item['record_id']: item for item in progress['results']}

for index, record in enumerate(records):
    rid = record_id(record)
    if rid in results_by_id:
        print(f'Skipping completed record {index + 1}/30: {rid}')
        continue
    started = time.perf_counter()
    raw_response, request_error = '', None
    try:
        prompt = tokenizer.apply_chat_template(record['messages'][:2], tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=3072).to('cuda')
        torch.manual_seed(42)
        with torch.inference_mode():
            generated = model.generate(
                **inputs, max_new_tokens=1800, do_sample=True, temperature=0.15, top_p=0.9,
                pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
            )
        raw_response = tokenizer.decode(generated[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    except Exception as error:
        request_error = f'{type(error).__name__}: {error}'
    expected = json.loads(record['messages'][2]['content'])
    result = {
        'record_id': rid,
        'expected_role': expected.get('title'),
        'latency_ms': round((time.perf_counter() - started) * 1000, 2),
        'raw_response': raw_response,
        'parsed': extract_json(raw_response) if raw_response else None,
        'request_error': request_error,
    }
    results_by_id[rid] = result
    progress['results'] = list(results_by_id.values())
    progress['updated_at'] = datetime.now(timezone.utc).isoformat()
    temporary = PROGRESS_PATH.with_suffix('.tmp')
    temporary.write_text(json.dumps(progress, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(PROGRESS_PATH)
    del inputs
    if 'generated' in locals():
        del generated
    torch.cuda.empty_cache()
    print(f"Completed {index + 1}/30: {rid}; parsed={result['parsed'] is not None}; error={request_error}")

ordered = [results_by_id[record_id(record)] for record in records if record_id(record) in results_by_id]
print(f'Completed results: {len(ordered)}/30')
if len(ordered) != 30:
    raise RuntimeError('Evaluation is incomplete. Re-run this cell to resume before exporting.')

## Export raw adapter outputs for local scoring

Return the downloaded JSON to Codex. Do not merge or convert the adapter yet.

In [ ]:
final_report = dict(
    metadata,
    created_at=datetime.now(timezone.utc).isoformat(),
    gpu=gpu.name,
    package_versions={name: version(name) for name in ('torch', 'transformers', 'peft', 'accelerate', 'bitsandbytes')},
    results=ordered,
)
OUTPUT_PATH = Path('/content/adapter_holdout_outputs.json')
OUTPUT_PATH.write_text(json.dumps(final_report, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Output SHA-256: {sha256(OUTPUT_PATH)}')
files.download(str(OUTPUT_PATH))